In [1]:
!hostname

gpua026.delta.ncsa.illinois.edu


# MaxToki Stage 2 finetune prep — peak-aware sparsity mask

Pick up where `atac_rna_cell_pairing.ipynb` left off. That notebook produces a single **prep bundle** with:

* `rna_sub.h5ad` — RNA AnnData subset to MaxToki vocab and sorted by token id (obs has `age_categorical` + `sample`; var has `ensg` + `maxtoki_token`).
* `mome.h5mu` — MuData (RNA + ATAC), with `atac.X = X_peaks_rna` (per-RNA-cell soft peak accessibility imputed via sample-stratified GW-OT).
* `P2G.npz`, `X_peaks_rna.npz`, `reg_mass.npz` — sparse arrays the sparsity-bias builder needs.
* `manifest.json` — recipe + provenance (OT stratification, P2G window/decay, etc.).

## Stage 2 finetune intent

MaxToki has two pretraining stages (Theodoris/NVIDIA, bioRxiv 10.64898/2026.03.30.715396):

* **Stage 1** — single-cell next-token CE on RVE; gives the 217M / 1B backbones their gene–gene SDPA. Pretrained, frozen, **untouched here**.
* **Stage 2** — multi-cell *temporal* finetune on cell paragraphs:

  ```
  [<bos>, ctx_1_genes, <eos>, ..., <bos>, ctx_K_genes, <eos>,
   <boq>, query_genes, <eoq>, dummy_numeric]
  ```

  with **two heads** trained jointly:
  - **NextCell** — cross-entropy on the query's RVE tokens (LM loss inside `<boq>...<eoq>`).
  - **TimeBetweenCells** — MSE on a scalar predicted at the numeric position, against the time-lapse between the last context cell and the query cell.

We zero-shot scored these SKM samples already (`maxtoki-perturb`, status row 1). Now we **finetune Stage 2** on the same samples + one new ingredient: a per-cell **chromatin-aware attention sparsity mask**, added to every transformer layer's SDPA. Backbone weights stay frozen; gradients flow only through the LM head, the TimeBetween head, and an optional LoRA adapter.

Sparsity rule (the same per-cell key-mask now stitched across all `ctx + query` token positions of the multi-cell trajectory):

```
mask[t, j] = True   if  gene_at_pos_j is "open" in source-cell(j)
                       (reg_mass[c, gene_pos[j]]  >=  threshold[sample(c)])
                    OR  position j is a special token (<bos>/<eos>/<boq>/<eoq>/numeric)
mask[t, j] = False  otherwise   ->   add -inf at this *key* column inside SDPA
```

For finetune the bias is computed at trajectory-assembly time by **concatenating each cell's `keep_mask`** in order, so a single attendance vector spans every position of a cell paragraph.

Threshold rule (per donor sample *s*):

```
threshold[s] = multiplier * median( reg_mass[cells_in_s, :]  > 0 )
```

`multiplier=1.0` (default) keeps ~half the nonzero gene tokens per cell as keys. `min_keep=32` is a safety floor that takes top-K-by-mass when threshold gates too aggressively.

This notebook starts by loading the prep bundle below, then RVE-tokenizes each cell, builds the per-cell key-mask, writes the finetune bundle (`maxtoki_finetune_skm/`) the maxtoki-perturb Stage 2 trainer consumes, and ends with the trajectory assembler reference.

In [ ]:
# Load the prep bundle from atac_rna_cell_pairing.ipynb.
import json, os, numpy as np, scipy.sparse as sp, anndata as ad, mudata as mu
from pathlib import Path

PREP_DIR = Path(os.environ.get(
    "FIREFATE_PREP_DIR",
    "/projects/bhdw/asachan/tmp/atac_rna_pairing_skm_prep",
))
prep_manifest = json.loads((PREP_DIR / "manifest.json").read_text())
print(f"prep bundle: {PREP_DIR}")
print(f"  n_cells={prep_manifest['n_cells']}  "
      f"n_genes={prep_manifest['n_genes']}  "
      f"n_peaks={prep_manifest['n_peaks']}")

rna_sub     = ad.read_h5ad(PREP_DIR / "rna_sub.h5ad")
mome        = mu.read_h5mu(PREP_DIR / "mome.h5mu")
P2G         = sp.load_npz (PREP_DIR / "P2G.npz")
X_peaks_rna = sp.load_npz (PREP_DIR / "X_peaks_rna.npz")
reg_mass    = sp.load_npz (PREP_DIR / "reg_mass.npz")

out_tmp = os.environ.get("OUT_TMP", "/projects/bhdw/asachan/tmp")
print(f"loaded:  rna_sub={rna_sub.shape}  P2G={P2G.shape}  "
      f"X_peaks_rna={X_peaks_rna.shape}  reg_mass={reg_mass.shape}")

In [ ]:
# torch_pipeline tokenizer (no MLX dep) — same RVE algorithm as maxtoki_mlx.
# It picks up token_dictionary.json + gene_median.json from
#   <repo>/src/maxtoki_mlx/resources/  (same files atac_rna_cell_pairing.ipynb references when subsetting RNA to MaxToki vocab).
# To use the temporal-aware dict (with <boq>/<eoq>/numeric tokens) for
# TimeBetweenCells finetune later, set MAXTOKI_TOKEN_DICT in the env.
import sys
TORCH_PIPE = "/projects/bhdw/asachan/methods/maxtoki-perturb/scripts/torch_pipeline"
if TORCH_PIPE not in sys.path:
    sys.path.insert(0, TORCH_PIPE)

from tokenizer import CellTokenizer, MODEL_INPUT_SIZE  # type: ignore

tok = CellTokenizer()
print(f"vocab={len(tok.token_dict)}  genes_with_median={tok.num_genes}  "
      f"bos={tok.bos_id}  eos={tok.eos_id}  model_input_size={MODEL_INPUT_SIZE}  "
      f"has_temporal_tokens={tok.has_temporal_tokens}")

In [ ]:
# rna_sub was vocab-subset in atac_rna_cell_pairing.ipynb, but the tokenizer additionally needs a
# gene_median entry per ENSG. Verify alignment.
ensg = rna_sub.var["ensg"].astype(str).values
in_tok = np.array([tok.has_gene(e) for e in ensg])
print(f"rna_sub genes accepted by MaxToki tokenizer: {in_tok.sum():,}/{len(in_tok):,} "
      f"({in_tok.mean():.1%})")
if not in_tok.all():
    drop = ensg[~in_tok][:5]
    print(f"  example dropped (no median): {list(drop)}")

In [ ]:
# RVE tokenize each cell while remembering the gene_idx (in rna_sub.var) at every
# token position. The rank logic mirrors CellTokenizer.tokenize_expression.
import numpy as np, scipy.sparse as sp
from tqdm.auto import tqdm

def rve_tokenize_with_positions(adata, tok, max_len=MODEL_INPUT_SIZE,
                                count_layer="counts", target_sum=1e4):
    """Returns parallel python lists, one entry per cell:
        input_ids[c]      : list[int]  -- [<bos>, gene_tok_1, ..., <eos>]
        gene_positions[c] : list[int]  -- gene_idx in adata.var (or -1 for specials)
        n_counts[c]       : float
    The gene_idx-per-position arrays are what the dataloader uses at train time
    to slice P2G when building the bias for each cell."""
    X = adata.layers[count_layer] if count_layer in adata.layers else adata.X
    X = sp.csr_matrix(X) if not sp.issparse(X) else X.tocsr()
    ensg = adata.var["ensg"].astype(str).values

    tok_id = np.full(adata.n_vars, -1, dtype=np.int64)
    median = np.full(adata.n_vars, np.nan, dtype=np.float64)
    for j, e in enumerate(ensg):
        if tok.has_gene(e):
            tok_id[j] = tok.gene_token(e)
            median[j] = tok.gene_median[e]
    keep = tok_id >= 0
    bos, eos = tok.bos_id, tok.eos_id

    ids_out, pos_out, n_out = [], [], []
    for i in tqdm(range(adata.n_obs), desc="RVE tokenize"):
        row = X[i].toarray().ravel().astype(np.float64)
        nc  = float(row.sum())
        if nc <= 0:
            ids_out.append([bos, eos]); pos_out.append([-1, -1]); n_out.append(0.0); continue
        scores = (row / nc) * target_sum
        m = keep & (row > 0)
        idx = np.where(m)[0]
        scores = scores[idx] / median[idx]
        order  = np.argsort(-scores, kind="stable")
        ranked = idx[order]
        if ranked.size > max_len - 2:
            ranked = ranked[:max_len-2]
        ids_out.append(np.concatenate([[bos], tok_id[ranked], [eos]]).astype(np.int64).tolist())
        pos_out.append(np.concatenate([[-1],  ranked,         [-1]]).astype(np.int64).tolist())
        n_out.append(nc)
    return ids_out, pos_out, n_out

input_ids, gene_positions, n_counts = rve_tokenize_with_positions(rna_sub, tok)
seq_lens = np.array([len(x) for x in input_ids])
print(f"seq_len  min/median/p95/max = "
      f"{seq_lens.min()}/{int(np.median(seq_lens))}/{int(np.percentile(seq_lens,95))}/{seq_lens.max()}")
print(f"cells with truncated RVE (=={MODEL_INPUT_SIZE}): "
      f"{(seq_lens == MODEL_INPUT_SIZE).sum()}/{len(seq_lens)}")

In [ ]:
# Reference key-mask builder — this is the function the maxtoki-perturb
# finetune dataloader imports verbatim.
import numpy as np, scipy.sparse as sp

# reg_mass = (n_cells, n_genes_rna_sub).  Cell 46 already computed this for
# the volcano; recompute defensively in case this section is run standalone.
if "reg_mass" not in dir():
    X_open   = mome["atac"].X.toarray() if sp.issparse(mome["atac"].X) else mome["atac"].X
    reg_mass = (X_open @ P2G).astype(np.float32)
reg_mass_csr = sp.csr_matrix(reg_mass)                 # streaming-friendly

def build_attn_keep_mask(cell_i, gene_pos, reg_mass_csr, threshold,
                         min_keep=32):
    """Per-cell binary key-mask over RVE token positions.

    Returns a bool array of length len(gene_pos):
        True  -> token is attendable as a key (special token, OR gene with
                 reg_mass >= threshold, OR fallback top-`min_keep` by mass)
        False -> mask out (add -inf at this key column inside SDPA).

    Specials (<bos>/<eos>, gene_pos == -1) are always kept."""
    gp    = np.asarray(gene_pos, dtype=np.int64)
    keep  = np.ones(len(gp), dtype=bool)
    valid = gp >= 0
    if not valid.any():
        return keep                                    # only specials
    mass_row    = np.asarray(reg_mass_csr[cell_i].todense()).ravel()
    mass_at_pos = mass_row[gp[valid]]
    keep_pos    = mass_at_pos >= threshold
    # min_keep floor: ensure at least min_keep gene-tokens stay attendable
    if keep_pos.sum() < min_keep and len(mass_at_pos) >= min_keep:
        order = np.argsort(-mass_at_pos)               # high mass first
        floor = np.zeros_like(keep_pos)
        floor[order[:min_keep]] = True
        keep_pos = keep_pos | floor
    keep[valid] = keep_pos
    return keep

def keep_mask_to_attn_bias(keep, dtype=np.float32):
    """Boolean keep-mask -> additive (seq_len,) bias for SDPA / flash-attn:
    0 at attendable keys, -inf at masked keys.  Broadcast across queries."""
    bias = np.zeros(len(keep), dtype=dtype)
    bias[~keep] = -np.inf
    return bias

# Per-sample thresholds: median of nonzero reg_mass entries within each donor.
THRESH_MULTIPLIER = 1.0          # tune ~ in [0.5, 2.0] to control sparsity
samples_arr = rna_sub.obs["sample"].astype(str).values
thresholds  = {}
for s in sorted(np.unique(samples_arr)):
    sub = reg_mass[samples_arr == s]
    nz  = sub[sub > 0]
    thresholds[s] = float(THRESH_MULTIPLIER * np.median(nz)) if nz.size else 0.0
print("per-sample reg_mass thresholds (median of nonzero entries x "
      f"{THRESH_MULTIPLIER}):")
for s, t in thresholds.items():
    print(f"  {s:>8s}: {t:.4f}")

In [ ]:
# Demo 1: attendable fraction per cell (how sparse did we get?)
keep_frac = np.zeros(rna_sub.n_obs, dtype=np.float32)
for c in range(rna_sub.n_obs):
    keep = build_attn_keep_mask(c, gene_positions[c], reg_mass_csr,
                                thresholds[samples_arr[c]])
    keep_frac[c] = keep.mean()
print(f"attendable-fraction per cell  "
      f"p5={np.percentile(keep_frac,5):.2f}  "
      f"median={np.median(keep_frac):.2f}  "
      f"p95={np.percentile(keep_frac,95):.2f}")

# Demo 2: biology check — does the kept gene set differ between young and old?
# Aggregate per-gene "kept fraction" within each age bucket.
ages = rna_sub.obs["age_categorical"].astype(str).values
unique_ages = sorted(np.unique(ages))
if len(unique_ages) >= 2:
    a_y, a_o = unique_ages[0], unique_ages[-1]
    kept_y = np.zeros(rna_sub.n_vars, dtype=np.int64)
    kept_o = np.zeros(rna_sub.n_vars, dtype=np.int64)
    for c in range(rna_sub.n_obs):
        keep = build_attn_keep_mask(c, gene_positions[c], reg_mass_csr,
                                    thresholds[samples_arr[c]])
        gp = np.asarray(gene_positions[c], dtype=np.int64)
        valid = gp >= 0
        if ages[c] == a_y:
            kept_y[gp[valid][keep[valid]]] += 1
        elif ages[c] == a_o:
            kept_o[gp[valid][keep[valid]]] += 1
    n_y, n_o = max(1, (ages == a_y).sum()), max(1, (ages == a_o).sum())
    delta = kept_o.astype(np.float64) / n_o - kept_y.astype(np.float64) / n_y
    print(f"\ngenes kept much more often in {a_o} than {a_y}:")
    for g in np.argsort(-delta)[:10]:
        print(f"  {rna_sub.var_names.values[g]:18s}  "
              f"keep_frac[{a_o}]={kept_o[g]/n_o:.2f}  "
              f"keep_frac[{a_y}]={kept_y[g]/n_y:.2f}  delta={delta[g]:+.2f}")
    print(f"\ngenes kept much more often in {a_y} than {a_o}:")
    for g in np.argsort( delta)[:10]:
        print(f"  {rna_sub.var_names.values[g]:18s}  "
              f"keep_frac[{a_y}]={kept_y[g]/n_y:.2f}  "
              f"keep_frac[{a_o}]={kept_o[g]/n_o:.2f}  delta={delta[g]:+.2f}")

In [ ]:
# Write the finetune bundle.
import json, scipy.sparse as sp
from pathlib import Path

bundle = Path(out_tmp) / "maxtoki_finetune_skm"
bundle.mkdir(parents=True, exist_ok=True)

# 1. sparse arrays for transparency / on-the-fly mask reconstruction
sp.save_npz(bundle / "reg_mass.npz",    reg_mass_csr)
sp.save_npz(bundle / "P2G.npz",         P2G.tocsr())
sp.save_npz(bundle / "X_peaks_rna.npz", sp.csr_matrix(X_peaks_rna))

# 2. cells.parquet — one row per cell, with a precomputed keep_mask so the
#    Stage 2 trajectory assembler can just concatenate per-cell pieces.
keep_masks = []
for c in range(rna_sub.n_obs):
    keep = build_attn_keep_mask(c, gene_positions[c], reg_mass_csr,
                                thresholds[samples_arr[c]])
    keep_masks.append(keep.astype(bool).tolist())

try:
    import pyarrow as pa, pyarrow.parquet as pq

    cells_tbl = pa.table({
        "cell_id":         pa.array([str(x) for x in rna_sub.obs_names.values]),
        "input_ids":       pa.array(input_ids,      type=pa.list_(pa.int64())),
        "gene_positions":  pa.array(gene_positions, type=pa.list_(pa.int64())),
        "keep_mask":       pa.array(keep_masks,     type=pa.list_(pa.bool_())),
        "n_counts":        pa.array(n_counts,       type=pa.float64()),
        "age_categorical": pa.array([str(x) for x in rna_sub.obs["age_categorical"].values]),
        "sample":          pa.array([str(x) for x in rna_sub.obs["sample"].values]),
    })
    pq.write_table(cells_tbl, bundle / "cells.parquet")

    gene_tbl = pa.table({
        "gene_idx":      pa.array(np.arange(rna_sub.n_vars, dtype=np.int64)),
        "ensg":          pa.array(rna_sub.var["ensg"].astype(str).values),
        "symbol":        pa.array(rna_sub.var_names.astype(str).values),
        "maxtoki_token": pa.array(rna_sub.var["maxtoki_token"].astype(np.int64).values),
    })
    pq.write_table(gene_tbl, bundle / "genes.parquet")

    av = mome["atac"].var
    peak_tbl = pa.table({
        "peak_idx": pa.array(np.arange(av.shape[0], dtype=np.int64)),
        "chrom":    pa.array(av["chrom"].astype(str).values),
        "start":    pa.array(av["start"].astype(np.int64).values),
        "end":      pa.array(av["end"].astype(np.int64).values),
    })
    pq.write_table(peak_tbl, bundle / "peaks.parquet")
except ImportError:
    import pickle
    pickle.dump({"input_ids": input_ids, "gene_positions": gene_positions,
                 "keep_mask": keep_masks, "n_counts": n_counts,
                 "cell_id":  rna_sub.obs_names.astype(str).tolist(),
                 "age":      rna_sub.obs["age_categorical"].astype(str).tolist(),
                 "sample":   rna_sub.obs["sample"].astype(str).tolist()},
                open(bundle / "cells.pkl", "wb"))

# 3. manifest — recipe + paths the Stage 2 finetune script needs
TOK_DICT_PATH = "/projects/bhdw/asachan/methods/maxtoki-perturb/src/maxtoki_mlx/resources/token_dictionary.json"
GENE_MED_PATH = "/projects/bhdw/asachan/methods/maxtoki-perturb/src/maxtoki_mlx/resources/gene_median.json"

manifest = {
    "n_cells":          int(rna_sub.n_obs),
    "n_genes":          int(rna_sub.n_vars),
    "n_peaks":          int(mome["atac"].n_vars),
    "model_input_size": int(MODEL_INPUT_SIZE),
    "tokenizer": {
        "token_dict":  TOK_DICT_PATH,
        "gene_median": GENE_MED_PATH,
        "vocab_size":  int(len(tok.token_dict)),
        "bos_id":      int(tok.bos_id),
        "eos_id":      int(tok.eos_id),
        "has_temporal_tokens": bool(tok.has_temporal_tokens),
    },
    "ot": {
        "stratification": "sample (per-donor GW couplings, sample-block-diagonal coupling enforced)",
        "T_dir":          str(Path(out_tmp) / "T_npy_sample_matched"),
    },
    "p2g": {
        "win_bp":   250_000,
        "scale_bp": 50_000,
        "decay":    "exp(-d/scale)",
        "shape":    list(P2G.shape),
        "nnz":      int(P2G.nnz),
    },
    "attn_sparsity": {
        "intent":          "freeze 217M backbone gene-gene SDPA; finetune ADDS a peak-aware key-mask so each cell only attends to chromatin-open genes",
        "rule":            "additive bias  -inf at key positions where gene at that position has reg_mass < sample_threshold; 0 elsewhere; specials always attendable",
        "threshold_recipe":"per-sample median of nonzero reg_mass entries (cells_in_s x genes_rna_sub)",
        "multiplier":      float(THRESH_MULTIPLIER),
        "min_keep":        32,
        "thresholds":      {str(s): float(t) for s, t in thresholds.items()},
        "keep_frac_p5":    float(np.percentile(keep_frac, 5)),
        "keep_frac_p50":   float(np.median(keep_frac)),
        "keep_frac_p95":   float(np.percentile(keep_frac, 95)),
    },
    "stage2_finetune": {
        "stage":             "stage_2_temporal_finetune",
        "trajectory_format": "[<bos>, ctx_1, <eos>, ..., <bos>, ctx_K, <eos>, <boq>, query, <eoq>, dummy_numeric]",
        "default_K_context": 2,
        "max_seq_length":    16384,
        "loss_heads": [
            {
                "name":       "NextCell",
                "loss":       "cross_entropy",
                "target":     "query gene tokens (positions [query_start : query_end])",
                "weight":     1.0,
            },
            {
                "name":       "TimeBetweenCells",
                "loss":       "mse",
                "target":     "time_lapse_target (years between last context cell and query cell), regressed from hidden state at numeric_position",
                "weight":     0.1,
            },
        ],
        "frozen":               "217M backbone weights (Stage 1 pretrained); finetune updates LM head + TimeBetween head + optional LoRA adapter on attention",
        "sparsity_application": "additive (B,1,1,Lk) attn_bias broadcast over all SDPA layers; -inf at positions where keep_mask is False, 0 elsewhere",
        "tokenizer_requirement":"full BioNeMo token_dictionary_v1.json with <boq>, <eoq>, numeric tokens; export MAXTOKI_TOKEN_DICT before running the finetune script",
        "trajectory_assembler": "see assemble_trajectory_for_finetune() in this notebook; not pre-materialized — sampled per epoch by sample_trajectory_specs()",
    },
    "files": {
        "reg_mass":    "reg_mass.npz     (csr  n_cells x n_genes)  -- per-cell per-gene peak mass; source of the mask",
        "P2G":         "P2G.npz          (csr  n_peaks x n_genes)",
        "X_peaks":     "X_peaks_rna.npz  (csr  n_cells x n_peaks)",
        "cells":       "cells.parquet    (one row per cell, with input_ids, gene_positions, keep_mask, age, sample)",
        "genes":       "genes.parquet",
        "peaks":       "peaks.parquet",
    },
}
(bundle / "manifest.json").write_text(json.dumps(manifest, indent=2))
print("bundle:", bundle)
for p in sorted(bundle.iterdir()):
    print(f"  {p.name}  ({p.stat().st_size/1e6:.1f} MB)")

### Hand-off to maxtoki-perturb Stage 2 finetune

Two atomic inputs to the trainer:
* `cells.parquet` — per-cell `input_ids`, `gene_positions`, `keep_mask`, `age`, `sample`
* per-epoch trajectory specs (sampled at runtime, not pre-saved) — `(query_idx, context_idxs, time_lapse)`

Pseudocode for one batch:

```python
import pyarrow.parquet as pq, json, numpy as np, torch, torch.nn.functional as F

bundle   = "/projects/bhdw/asachan/tmp/maxtoki_finetune_skm"
manifest = json.load(open(f"{bundle}/manifest.json"))
cells    = pq.read_table(f"{bundle}/cells.parquet").to_pandas()

# Sample fresh trajectory specs each epoch:
specs = sample_trajectory_specs(cells, K=manifest["stage2_finetune"]["default_K_context"])

# Per-row at __getitem__ time:
def make_row(spec):
    traj = assemble_trajectory_for_finetune(
        cells,
        query_idx     = spec["query_idx"],
        context_idxs  = spec["context_idxs"],
        time_lapse    = spec["time_lapse"],
        tok           = tok,
        seq_length    = manifest["stage2_finetune"]["max_seq_length"],
    )
    ids   = np.asarray(traj["input_ids"], dtype=np.int64)
    keep  = np.asarray(traj["keep_mask"], dtype=bool)
    bias  = np.where(keep, 0.0, -np.inf).astype(np.float32)   # (L,) -> broadcasts to (B,1,1,Lk)
    return {
        "input_ids":         ids,
        "attn_bias":         bias,
        "query_start":       traj["query_start"],
        "query_end":         traj["query_end"],
        "numeric_position":  traj["numeric_position"],
        "time_lapse_target": traj["time_lapse_target"],
    }

# Multi-task loss:  L = w_CE * NextCell_CE  +  w_MSE * TimeBetweenCells_MSE
def loss_fn(out, batch, w_ce=1.0, w_mse=0.1):
    # NextCell CE on query positions only (causal-LM shift)
    logits = out.lm_head_logits                                      # (B, L, V)
    shift_logits = logits[:, batch.query_start - 1 : batch.query_end - 1]
    shift_labels = batch.input_ids[:, batch.query_start : batch.query_end]
    ce  = F.cross_entropy(shift_logits.transpose(1, 2), shift_labels)
    # TimeBetweenCells MSE on hidden state at numeric_position
    h_t = out.hidden_states[:, batch.numeric_position]
    mse = F.mse_loss(out.time_head(h_t).squeeze(-1), batch.time_lapse_target)
    return w_ce * ce + w_mse * mse

# Inside every transformer layer's SDPA call, fold the bias:
#   logits = (Q @ K.T) / sqrt(d) + attn_bias[None, None, None, :]
# Backbone (217M Stage 1) frozen; only LM head + time_head + LoRA adapters get gradient.
```

### Knobs to sweep

* `THRESH_MULTIPLIER` (see `build_attn_keep_mask` below) — sparsity strength (0.5× looser, 2× stricter).
* `min_keep` (see `build_attn_keep_mask` below) — floor on attendable gene tokens per cell.
* `K_context` (see assembler below) — number of context cells per trajectory; 2–3 fits `seq_length=16384`.
* `w_ce / w_mse` — multitask weights; `w_mse=0.1` is a typical starting point so MSE doesn't dominate CE.

## Stage 2 trajectory assembly reference

For Stage 2 finetune the dataloader needs to glue together `K` context cells + 1 query cell from `cells.parquet` into a single multi-cell trajectory, and concatenate their `keep_mask`s position-wise. The assembler below is the function the maxtoki-perturb finetune script imports verbatim — it runs in `O(seq_len)` and returns everything the multitask loss needs (`input_ids`, per-position provenance, `keep_mask`, `query_start/end`, `numeric_position`, `time_lapse_target`).

The basic packaged `token_dictionary.json` does **not** include `<boq>`, `<eoq>`, or numeric tokens. Set `MAXTOKI_TOKEN_DICT` to the full BioNeMo `token_dictionary_v1.json` (e.g. the one packaged inside the distcp checkpoint) before running this section, otherwise `tok.has_temporal_tokens` is `False` and the assembler will refuse to run.

In [ ]:
# Stage 2 trajectory assembler.  Stitches K context cells + 1 query cell from
# cells.parquet into a single multi-cell finetune row, aligning keep_mask
# across positions.
from tokenizer import pick_dummy_numeric_token  # type: ignore


def assemble_trajectory_for_finetune(
    cells_df, query_idx, context_idxs, time_lapse, tok,
    seq_length=16384,
):
    """Build one Stage 2 finetune row.

    Returns dict with:
        input_ids:         list[int]   length L (<= seq_length)
        cell_at_position:  list[int]   rna_sub.obs index, or query_idx for query specials
        gene_at_position:  list[int]   rna_sub.var index, or -1 for non-gene tokens
        keep_mask:         list[bool]  per-position attention attendability
        query_start:       int         index of first query gene token (NextCell CE start)
        query_end:         int         index of <eoq>           (NextCell CE end, exclusive)
        numeric_position:  int         index of dummy numeric   (TimeBetweenCells MSE read-out)
        time_lapse_target: float       MSE label
    """
    if not tok.has_temporal_tokens:
        raise RuntimeError(
            "Tokenizer lacks <boq>/<eoq>/numeric tokens. Set MAXTOKI_TOKEN_DICT to "
            "the full BioNeMo token_dictionary_v1.json (packaged inside the distcp "
            "checkpoint) before running Stage 2 prep."
        )

    boq_id  = tok.boq_id
    eoq_id  = tok.eoq_id
    num_tok = pick_dummy_numeric_token(tok)

    ids, cells_at, genes_at, keep_at = [], [], [], []

    # Context cells: keep [<bos>, gene_tokens, <eos>] verbatim from cells.parquet
    for c in context_idxs:
        c_ids = list(cells_df.input_ids.iloc[c])
        c_gp  = list(cells_df.gene_positions.iloc[c])
        c_km  = list(cells_df.keep_mask.iloc[c])
        ids.extend(c_ids)
        genes_at.extend(c_gp)
        keep_at.extend(c_km)
        cells_at.extend([int(c)] * len(c_ids))

    # Query cell: drop its own <bos>/<eos>, wrap its gene tokens in <boq>/<eoq>
    q_ids = list(cells_df.input_ids.iloc[query_idx])
    q_gp  = list(cells_df.gene_positions.iloc[query_idx])
    q_km  = list(cells_df.keep_mask.iloc[query_idx])
    if len(q_ids) >= 2 and q_ids[0] == tok.bos_id and q_ids[-1] == tok.eos_id:
        q_ids, q_gp, q_km = q_ids[1:-1], q_gp[1:-1], q_km[1:-1]

    ids.append(boq_id);  cells_at.append(int(query_idx)); genes_at.append(-1); keep_at.append(True)
    query_start = len(ids)
    ids.extend(q_ids);   cells_at.extend([int(query_idx)] * len(q_ids))
    genes_at.extend(q_gp); keep_at.extend(q_km)
    query_end = len(ids)
    ids.append(eoq_id);  cells_at.append(int(query_idx)); genes_at.append(-1); keep_at.append(True)
    numeric_position = len(ids)
    ids.append(num_tok); cells_at.append(int(query_idx)); genes_at.append(-1); keep_at.append(True)

    # Truncate from the front if over budget; drop earliest context cells.
    if len(ids) > seq_length:
        eos_id = tok.eos_id
        excess = len(ids) - seq_length
        cut    = 0
        for j, t in enumerate(ids):
            if t == eos_id and j + 1 >= excess:
                cut = j + 1
                break
        if cut == 0:                       # fallback: hard token-level trim
            cut = excess
        ids       = ids[cut:]
        cells_at  = cells_at[cut:]
        genes_at  = genes_at[cut:]
        keep_at   = keep_at[cut:]
        query_start      -= cut
        query_end        -= cut
        numeric_position -= cut

    return {
        "input_ids":         ids,
        "cell_at_position":  cells_at,
        "gene_at_position":  genes_at,
        "keep_mask":         keep_at,
        "query_start":       query_start,
        "query_end":         query_end,
        "numeric_position":  numeric_position,
        "time_lapse_target": float(time_lapse),
    }


def sample_trajectory_specs(cells_df, K=2, n_traj=None,
                            age_col="age_categorical",
                            age_to_pseudotime=None, seed=0):
    """For each query cell with at least K earlier-age cells, pick K context
    cells (one per distinct earlier age when possible).  Returns list of
    dicts: {query_idx, context_idxs, time_lapse}."""
    rng = np.random.default_rng(seed)
    ages_str    = cells_df[age_col].astype(str).values
    unique_ages = sorted(np.unique(ages_str))
    if age_to_pseudotime is None:
        try:
            age_to_pseudotime = {a: float(a) for a in unique_ages}
        except ValueError:
            age_to_pseudotime = {a: float(i) for i, a in enumerate(unique_ages)}
    pt = np.array([age_to_pseudotime[a] for a in ages_str])

    candidate_idxs = np.arange(len(cells_df))
    if n_traj is not None and n_traj < len(candidate_idxs):
        candidate_idxs = rng.choice(candidate_idxs, size=n_traj, replace=False)

    specs = []
    for q in candidate_idxs:
        q_pt = pt[q]
        earlier = np.where(pt < q_pt)[0]
        if len(earlier) < K:
            continue
        earlier_ages   = pt[earlier]
        unique_earlier = np.unique(earlier_ages)
        if len(unique_earlier) >= K:
            picked = rng.choice(unique_earlier, size=K, replace=False)
            ctx = [int(rng.choice(earlier[earlier_ages == a])) for a in picked]
        else:
            ctx = rng.choice(earlier, size=K, replace=False).astype(int).tolist()
        last_ctx_pt = max(pt[c] for c in ctx)
        specs.append({
            "query_idx":    int(q),
            "context_idxs": [int(c) for c in ctx],
            "time_lapse":   float(q_pt - last_ctx_pt),
        })
    return specs


# Demo: build a 5-row mini-batch of trajectories from the SKM cells we just
# tokenized.  This proves the assembler aligns input_ids, per-position
# provenance, and the keep_mask across cells.
import pandas as pd

demo_cells_df = pd.DataFrame({
    "input_ids":      input_ids,
    "gene_positions": gene_positions,
    "keep_mask":      keep_masks if "keep_masks" in dir() else [
        build_attn_keep_mask(c, gene_positions[c], reg_mass_csr,
                             thresholds[samples_arr[c]]).astype(bool).tolist()
        for c in range(rna_sub.n_obs)
    ],
    "age_categorical": rna_sub.obs["age_categorical"].astype(str).values,
    "sample":          rna_sub.obs["sample"].astype(str).values,
})

specs = sample_trajectory_specs(demo_cells_df, K=2, n_traj=5, seed=42)
print(f"sampled {len(specs)} trajectory specs out of {rna_sub.n_obs} cells")

if specs and tok.has_temporal_tokens:
    s    = specs[0]
    traj = assemble_trajectory_for_finetune(
        demo_cells_df,
        query_idx     = s["query_idx"],
        context_idxs  = s["context_idxs"],
        time_lapse    = s["time_lapse"],
        tok           = tok,
        seq_length    = 16384,
    )
    print("\ndemo trajectory:")
    print(f"  query  cell_idx={s['query_idx']:>5d}  "
          f"age={demo_cells_df.age_categorical.iloc[s['query_idx']]:<5s}  "
          f"sample={demo_cells_df['sample'].iloc[s['query_idx']]}")
    print(f"  ctxs   cell_idxs={s['context_idxs']}  "
          f"ages={[demo_cells_df.age_categorical.iloc[c] for c in s['context_idxs']]}")
    print(f"  time_lapse_target={s['time_lapse']:.1f}  (years)")
    print(f"  L={len(traj['input_ids']):>5d}  "
          f"query_start={traj['query_start']:>5d}  "
          f"query_end={traj['query_end']:>5d}  "
          f"numeric_position={traj['numeric_position']:>5d}")
    print(f"  attendable_frac (specials always True): {np.mean(traj['keep_mask']):.2f}")
    # Sanity: lengths agree
    L = len(traj["input_ids"])
    assert len(traj["cell_at_position"]) == L
    assert len(traj["gene_at_position"]) == L
    assert len(traj["keep_mask"])        == L
    assert traj["input_ids"][traj["numeric_position"]] in tok.numeric_token_ids
    print("  ✓ shape + numeric-token sanity checks passed")
elif not tok.has_temporal_tokens:
    print("[skip demo] tokenizer has no temporal tokens. "
          "Set MAXTOKI_TOKEN_DICT=/path/to/token_dictionary_v1.json and rerun.")